# Notebook 01 — SPX Implied-Volatility Surface Calibration

Fetches a real SPX options chain from yfinance, constructs the 8×11 IV surface,
and calibrates the Rough Heston (Lifted) model using the Gauss-Newton surrogate.

**Runtime estimate:** 2–5 min (network + calibration)

In [ ]:
import os, sys
sys.path.insert(0, os.path.join(os.path.dirname(os.getcwd()), "src") if os.path.basename(os.getcwd()) == "notebooks"
                else os.path.join(os.getcwd(), "src"))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import torch

plt.rcParams.update({
    "figure.dpi": 120,
    "axes.spines.top": False,
    "axes.spines.right": False,
    "axes.labelsize": 11,
    "font.family": "serif",
})
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {DEVICE}")

from datetime import date
from market.spx_data import download_spx_chain, clean_chain, to_iv_surface, T_GRID, K_GRID
from calibrate import _load_normalizers
from calibrate_fast import calibrate_newton_h
from fno_model import MirrorPaddedFNO2d
from normalizers import ParameterNormalizer, IVSurfaceNormalizer


## 1. Load the FNO v3 Model and Normalizers

In [ ]:
model = MirrorPaddedFNO2d(param_dim=6).to(DEVICE)
model.load_state_dict(torch.load(
    "../artifacts/weights/fno_v3_final_prod.pth", map_location=DEVICE))
model.eval()
_load_normalizers("v3")
print("Model loaded — parameters:", sum(p.numel() for p in model.parameters()))


## 2. Download and Clean the SPX Options Chain

In [ ]:
SNAPSHOT_DATE = date(2024, 1, 2)   # change to any date with cached parquet
S0 = 4742.0   # SPX spot on 2024-01-02
R  = 0.053    # approx risk-free rate (early 2024)
Q  = 0.016    # approx SPX dividend yield
df_raw  = download_spx_chain(SNAPSHOT_DATE, cache=True)
df_clean = clean_chain(df_raw)
iv_surface = to_iv_surface(df_clean, S0, R, Q)   # shape (8, 11)

print(f"Options fetched:  {len(df_raw)}")
print(f"After cleaning:   {len(df_clean)}")
print(f"IV surface shape: {iv_surface.shape}")
print(f"ATM vol (T=0.6):  {iv_surface[2, 5]:.4f}  ({iv_surface[2, 5]*100:.2f}%)")


## 3. Plot the Market IV Surface

In [ ]:
from mpl_toolkits.mplot3d import Axes3D

K_abs = np.exp(K_GRID)          # strike / forward
T_mesh, K_mesh = np.meshgrid(T_GRID, K_abs, indexing="ij")

fig = plt.figure(figsize=(10, 5))
ax  = fig.add_subplot(111, projection="3d")
surf = ax.plot_surface(T_mesh, K_mesh, iv_surface * 100,
                        cmap="RdYlGn_r", alpha=0.9, linewidth=0)
ax.set_xlabel("Maturity T (years)")
ax.set_ylabel("Strike / Forward")
ax.set_zlabel("Implied Vol (%)")
ax.set_title(f"SPX IV Surface — {SNAPSHOT_DATE}")
fig.colorbar(surf, shrink=0.5, label="IV (%)")
plt.tight_layout()
plt.savefig("../tex/thesis/figures/fig_iv_surface.png", dpi=150, bbox_inches="tight")
plt.show()


## 4. Calibrate Rough Heston via Gauss-Newton

Uses the FNO v3 surrogate and exact Jacobians (`torch.func.jacfwd`).

In [ ]:
result = calibrate_newton_h(model, iv_surface, T_GRID, K_GRID, max_iter=20, verbose=True)

print("\nCalibration result:")
rmse_bps = np.sqrt(result['final_mse']) * 1e4
print(f"  RMSE : {rmse_bps:.1f} bps")
print(f"  iters: {result['n_iter']}")
for k in ['v0','zeta','lambda','sigma','rho','H']:
    print(f"  {k:6s} = {result[k]:.4f}")


## 5. Compare Market vs Model Surface

In [ ]:
pred_surface = result["iv_fitted"]              # (8, 11) in vol units
residuals    = (pred_surface - iv_surface) * 1e4   # in basis points

fig, axes = plt.subplots(1, 3, figsize=(15, 4))

for ax, data, title, cmap in zip(
        axes,
        [iv_surface * 100, pred_surface * 100, residuals],
        ["Market IV (%)", "Model IV (%)", "Residuals (bps)"],
        ["RdYlGn_r", "RdYlGn_r", "RdBu_r"]):
    im = ax.imshow(data, aspect="auto", cmap=cmap,
                   extent=[K_GRID[0], K_GRID[-1], T_GRID[-1], T_GRID[0]])
    ax.set_title(title)
    ax.set_xlabel("Log-moneyness")
    ax.set_ylabel("Maturity (years)")
    plt.colorbar(im, ax=ax, fraction=0.04)

plt.suptitle(f"SPX {SNAPSHOT_DATE}  |  RMSE = {np.sqrt(result['final_mse'])*1e4:.1f} bps", fontsize=12)
plt.tight_layout()
plt.show()
print(f"Max |residual|: {np.abs(residuals).max():.1f} bps")


## 6. Convergence History

In [ ]:
history = result.get("history", [])
if history:
    fig, ax = plt.subplots(figsize=(7, 4))
    ax.semilogy(range(1, len(history)+1), history, "o-", color="#2d6a9f")
    ax.set_xlabel("Iteration")
    ax.set_ylabel("RMSE (bps) — log scale")
    ax.set_title("Gauss-Newton convergence")
    ax.xaxis.set_major_locator(mticker.MaxNLocator(integer=True))
    plt.tight_layout()
    plt.show()
